<a href="https://colab.research.google.com/github/atomicSteiner/HealthcareSBERT/blob/main/Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#path for content
path_drive = "/content/drive/MyDrive/NLP/"
path_drive_red = "/content/"

In [ ]:
# Imports
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import os

In [ ]:
#use df
df = pd.read_csv(path_drive+"ohsumed_cleaned_train.csv")
df.head()

,seq_id,medline_ui,mesh_terms,title,abstract
0,1,87049087,Allied Health Personnel/*; Electric Countersho...,Refibrillation managed by EMT-Ds: incidence an...,Some patients converted from ventricular fibri...
1,1,87049088,"Antidepressive Agents, Tricyclic/*PO; Arrhythm...",Tricyclic antidepressant overdose: emergency d...,There is controversy regarding the appropriate...
2,1,87049089,Adult; Aircraft/*; Altitude/*; Blood Gas Monit...,Transconjunctival oxygen monitoring as a predi...,As the use of helicopters for air transport of...
3,1,87049090,Adolescence; Adult; Aged; Blood Glucose/*ME; D...,Serum glucose changes after administration of ...,A prospective clinical trial was conducted to ...
4,1,87049092,"Aged; Aged, 80 and over; Case Report; Female; ...",Nasogastric intubation: morbidity in an asympt...,An unusual case of a misdirected nasogastric t...


In [ ]:
# -----------------------------
# Improved MeSH parsing, prepare the data
# In this way we get lists from the initial mesh_terms column
# -----------------------------
def parse_mesh_improved(mesh_str):
    if not isinstance(mesh_str, str):
        return []

    terms = mesh_str.split(';')  # split by semicolon
    clean_terms = []

    for t in terms:
        t = t.strip()            # remove whitespace
        if t == '':
            continue
        # keep only main term (before '/')
        t = t.split('/')[0].strip()
        clean_terms.append(t)

    return clean_terms

df['mesh_terms'] = df['mesh_terms'].apply(parse_mesh_improved)

In [ ]:
print(df['mesh_terms'][0])

['Allied Health Personnel', 'Electric Countershock', 'Emergencies', 'Emergency Medical Technicians', 'Human', 'Prognosis', 'Recurrence', "Support, U.S. Gov't, P.H.S.", 'Time Factors', 'Transportation of Patients', 'Ventricular Fibrillation']


In [ ]:
embeddings_baseline = np.load(path_drive+"embeddings_baseline.npy")

In [ ]:
# Parameters
N_total = 300
N_mesh = 100
N_nn = 100
N_rand = 100
assert N_mesh + N_nn + N_rand == N_total

random.seed(42)
np.random.seed(42)

In [ ]:
# 1) MAP MeSH -> index list
from collections import defaultdict
mesh_to_idxs = defaultdict(list)
for idx, meshes in enumerate(df['mesh_terms']):
    for m in meshes:
        mesh_to_idxs[m].append(idx)

In [ ]:
# 2) Generate couples MeSH-overlap (avoiding duplicates)
mesh_pairs = set()
max_pairs_per_mesh = 20
for mesh, idxs in mesh_to_idxs.items():
    if len(idxs) < 2:
        continue
    sample = random.sample(idxs, min(len(idxs), max_pairs_per_mesh))
    for i in range(len(sample)):
        for j in range(i+1, len(sample)):
            a, b = sample[i], sample[j]
            pair = tuple(sorted((a,b)))
            mesh_pairs.add(pair)
mesh_pairs = list(mesh_pairs)
random.shuffle(mesh_pairs)
mesh_pairs = mesh_pairs[:N_mesh]
print("MeSH pairs:", len(mesh_pairs))

MeSH pairs: 100


In [ ]:
# 3) Generate couples NN (using embeddings) - for each index take the nearest neighbor not identical
nbrs = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='auto').fit(embeddings_baseline)
distances, indices = nbrs.kneighbors(embeddings_baseline)  # indices[i] contains itself + nearest
nn_pairs = set()
for i in range(len(embeddings_baseline)):
    for neigh_idx in indices[i][1:6]:  #not considering itself
        pair = tuple(sorted((i, int(neigh_idx))))
        # Avoid if already MeSH pair
        if pair in mesh_pairs:
            continue
        nn_pairs.add(pair)
nn_pairs = list(nn_pairs)
random.shuffle(nn_pairs)
nn_pairs = nn_pairs[:N_nn]
print("NN pairs:", len(nn_pairs))

NN pairs: 100


In [ ]:
# 4) Random pairs (no duplicates)
all_used = set(mesh_pairs) | set(nn_pairs)
rand_pairs = set()
n_doc = len(df)
while len(rand_pairs) < N_rand:
    a, b = np.random.randint(0,n_doc), np.random.randint(0,n_doc)
    if a == b:
        continue
    pair = tuple(sorted((a,b)))
    if pair in all_used or pair in rand_pairs:
        continue
    rand_pairs.add(pair)
rand_pairs = list(rand_pairs)
print("Random pairs:", len(rand_pairs))

Random pairs: 100


In [ ]:
# 5) Merge and create output dataframe
pairs_idx = mesh_pairs + nn_pairs + rand_pairs
types = (["mesh"]*len(mesh_pairs)) + (["nn"]*len(nn_pairs)) + (["rand"]*len(rand_pairs))

rows = []
for pid, (i,j) in enumerate(pairs_idx):
    rows.append({
        "pair_id": f"p{pid:04d}",
        "idx_a": i,
        "idx_b": j,
        "text_a": df.loc[i, "abstract"],
        "text_b": df.loc[j, "abstract"],
        "mesh_overlap": bool(set(df.loc[i,'mesh_terms']) & set(df.loc[j,'mesh_terms'])),
        "sample_type": types[pid]
    })
pairs_df = pd.DataFrame(rows)
print(pairs_df.head())
print("Total pairs:", len(pairs_df))


  pair_id  idx_a  idx_b                                             text_a  \
0   p0000  35847  36651  Cloned T cell lines specific for the antigen o...   
1   p0001  14260  29057  No substance should be administered unnecessar...   
2   p0002  10777  19089  The incidence of proximal femoral fractures in...   
3   p0003  35158  35412  Gamma interferon (IFN-gamma) and B cell stimul...   
4   p0004   6739  22959  Sixteen consecutive patients with tibial plate...   

                                              text_b  mesh_overlap sample_type  
0  BALB/c (H-2d) mice rendered tolerant to h-2b a...          True        mesh  
1  We vaccinated 244 newborn infants orally with ...          True        mesh  
2  The age distribution of anginose infectious mo...          True        mesh  
3  Macrophages are activated by lymphokines (LK) ...          True        mesh  
4  An investigation with respect to position of t...          True        mesh  
Total pairs: 300


In [ ]:
# Save couples file (to give to the annotators)
pairs_df.to_csv("annotation_pairs.csv", index=False, encoding='utf-8')

Istruzioni per annotatori:

Obiettivo: per ogni coppia di abstract (A, B) assegnare un punteggio di similarità 0–4:

- 4 = Equivalent (stesse informazioni / paraphrase)
- 3 = Highly related (stesso risultato/tema stretto)
- 2 = Related (stesso argomento, ma non stesso risultato)
- 1 = Marginally related (leggera connessione)
- 0 = Not related

Linee guida:
- Leggi entrambi gli abstract per intero.
- Scegli il punteggio che riflette meglio la relazione semantica.
- Se non sei sicuro, usa il valore più basso.
- Compila: pair_id, annotator_id (es. A1), score (0-4), comment (opzionale).

Esempi (impostare alcuni esempi concreti qui).


In [ ]:
#reread annotation_pairs.csv
pairs_df = pd.read_csv(path_drive+"annotation_pairs.csv")
pairs_df.head()

,pair_id,idx_a,idx_b,text_a,text_b,mesh_overlap,sample_type,human_mean
0,p0000,35847,36651,Cloned T cell lines specific for the antigen o...,BALB/c (H-2d) mice rendered tolerant to h-2b a...,True,mesh,2
1,p0001,14260,29057,No substance should be administered unnecessar...,We vaccinated 244 newborn infants orally with ...,True,mesh,1
2,p0002,10777,19089,The incidence of proximal femoral fractures in...,The age distribution of anginose infectious mo...,True,mesh,1
3,p0003,35158,35412,Gamma interferon (IFN-gamma) and B cell stimul...,Macrophages are activated by lymphokines (LK) ...,True,mesh,3
4,p0004,6739,22959,Sixteen consecutive patients with tibial plate...,An investigation with respect to position of t...,True,mesh,2


In [ ]:
df_sim_scores = pd.read_csv(path_drive+"similarity_scores.csv")
df_sim_scores.head()

,ID,similarity_score
0,p0000,1
1,p0001,2
2,p0002,1
3,p0003,2
4,p0004,2


In [ ]:
#add column df_sim_scores["Similarity_Score"] to the dataframe pairs_df
pairs_df["human_mean"] = df_sim_scores["similarity_score"]
pairs_df.head()

,pair_id,idx_a,idx_b,text_a,text_b,mesh_overlap,sample_type,human_mean
0,p0000,35847,36651,Cloned T cell lines specific for the antigen o...,BALB/c (H-2d) mice rendered tolerant to h-2b a...,True,mesh,1
1,p0001,14260,29057,No substance should be administered unnecessar...,We vaccinated 244 newborn infants orally with ...,True,mesh,2
2,p0002,10777,19089,The incidence of proximal femoral fractures in...,The age distribution of anginose infectious mo...,True,mesh,1
3,p0003,35158,35412,Gamma interferon (IFN-gamma) and B cell stimul...,Macrophages are activated by lymphokines (LK) ...,True,mesh,2
4,p0004,6739,22959,Sixteen consecutive patients with tibial plate...,An investigation with respect to position of t...,True,mesh,2


In [ ]:
#save the new file with the column of the scores
pairs_df.to_csv("annotation_pairs_with_human_scores.csv", index=False)

Now we can calculate baseline and finetuned similarities


In [ ]:
#import the necessary files
pairs = pd.read_csv(path_drive_red + "annotation_pairs_with_human_scores.csv")
baseline_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
ft_model = SentenceTransformer(path_drive + 'fine_tuned_sbert_ohsumed/fine_tuned_sbert_ohsumed')

In [ ]:
from scipy.stats import pearsonr, spearmanr, ttest_rel

In [ ]:
# 1 load the embeddings of the two models
base = np.load(path_drive + "embeddings_baseline.npy")      # shape: (n, dim)
ft = np.load(path_drive + "fine_tuned_embeddings.npy")

In [ ]:
print(base)

[[-0.02227223 -0.03670328  0.02825625 ... -0.02671325 -0.02989987
  -0.07183813]
 [ 0.02047469  0.06606434  0.01755299 ...  0.01555689  0.0217707
  -0.02892632]
 [ 0.0086581  -0.01430361 -0.0347599  ...  0.05006286 -0.05863208
  -0.06102149]
 ...
 [-0.03007015 -0.13517816 -0.01871574 ...  0.01007717  0.04351751
   0.03306811]
 [-0.02878388 -0.17532438  0.03797771 ...  0.03548967  0.05891364
  -0.00240316]
 [-0.04762145 -0.06326187  0.01079378 ...  0.02814896 -0.04049876
  -0.03028486]]


In [ ]:
print(ft)

[[ 0.01365097  0.02150222 -0.01308939 ...  0.02563861 -0.05731463
  -0.04078146]
 [ 0.04723598  0.06651387 -0.03484424 ... -0.02545564  0.02892508
  -0.03512208]
 [ 0.04281836  0.03444918 -0.07169809 ...  0.10198989 -0.13156196
  -0.04462488]
 ...
 [-0.0537833  -0.12354717  0.02569691 ...  0.04972714  0.04069616
  -0.00137861]
 [-0.03378206 -0.15613483  0.01290402 ...  0.05098518  0.04470451
   0.01073372]
 [-0.05399999 -0.0052386   0.01830168 ...  0.00526242 -0.03988857
   0.00991715]]


In [ ]:
#2 check that the index are integer
pairs["idx_a"] = pairs["idx_a"].astype(int)
pairs["idx_b"] = pairs["idx_b"].astype(int)

In [ ]:
# 3️ Calculate similarities for each couple
def cosine(a, b):
    return float(cosine_similarity([a], [b])[0][0])

In [ ]:
print("ft shape:", ft.shape)
print("max idx in pairs:", max(pairs["idx_a"].max(), pairs["idx_b"].max()))
print("min idx in pairs:", min(pairs["idx_a"].min(), pairs["idx_b"].min()))


ft shape: (36888, 384)
max idx in pairs: 36854
min idx in pairs: 161


In [ ]:
pairs["sim_base"] = [
    cosine(base[i1], base[i2]) for i1, i2 in zip(pairs["idx_a"], pairs["idx_b"])
]
pairs["sim_ft"] = [
    cosine(ft[i1], ft[i2]) for i1, i2 in zip(pairs["idx_a"], pairs["idx_b"])
]

In [ ]:
# 4️ Calculate the correlation with the human scores
pearson_base = pearsonr(pairs["human_mean"], pairs["sim_base"])
pearson_ft = pearsonr(pairs["human_mean"], pairs["sim_ft"])
spearman_base = spearmanr(pairs["human_mean"], pairs["sim_base"])
spearman_ft = spearmanr(pairs["human_mean"], pairs["sim_ft"])

In [ ]:
print(f"Pearson baseline: {pearson_base[0]:.3f}, fine-tuned: {pearson_ft[0]:.3f}")
print(f"Spearman baseline: {spearman_base[0]:.3f}, fine-tuned: {spearman_ft[0]:.3f}")

Pearson baseline: 0.598, fine-tuned: 0.563
Spearman baseline: 0.637, fine-tuned: 0.599


In [ ]:
# 5️ significative test
t_stat, p_val = ttest_rel(pairs["sim_ft"], pairs["sim_base"])
print(f"T-test: t = {t_stat:.3f}, p = {p_val:.4f}")

T-test: t = -4.670, p = 0.0000


In [ ]:
# 6️ save the results
pairs.to_csv("evaluation_similarity_results.csv", index=False)

other metrics

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, mean_absolute_error

# Create labels for the human imputs
threshold = 3
pairs['label'] = (pairs['human_mean'] >= threshold).astype(int)

# Calculate AUC
auc_base = roc_auc_score(pairs['label'], pairs['sim_base'])
auc_ft = roc_auc_score(pairs['label'], pairs['sim_ft'])

print(f"AUC baseline: {auc_base:.3f}, fine-tuned: {auc_ft:.3f}")


AUC baseline: 0.780, fine-tuned: 0.750


In [ ]:
# Calculate MSE and MAE
mse_base = mean_squared_error(pairs['human_mean'], pairs['sim_base'])
mse_ft = mean_squared_error(pairs['human_mean'], pairs['sim_ft'])

mae_base = mean_absolute_error(pairs['human_mean'], pairs['sim_base'])
mae_ft = mean_absolute_error(pairs['human_mean'], pairs['sim_ft'])

print(f"MSE baseline: {mse_base:.4f}, fine-tuned: {mse_ft:.4f}")
print(f"MAE baseline: {mae_base:.4f}, fine-tuned: {mae_ft:.4f}")


MSE baseline: 1.6925, fine-tuned: 1.7575
MAE baseline: 1.1713, fine-tuned: 1.1965
